## 0 · Config

In [1]:
import os, gc, glob, json, time, warnings, traceback
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

CKPT_DIR = "/kaggle/input/notebooks/shreyanshgoyaliitb/training/"     
COMP_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
OUT_CSV  = "/kaggle/working/submission.csv"

BATCH        = 4
NUM_WORKERS  = 4        
MAX_HOURS    = 8.0      
MIN_SLICES   = 8

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

ckpt_path = os.path.join(CKPT_DIR, "best.pt")
print("Loading ckpt")
ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
cfg = ck["cfg"]
BACKBONE   = cfg["backbone"]
IMG        = cfg["img"]
N_SLICES   = cfg["n_slices"]
PLANES     = cfg["planes"]
LABEL_COLS = cfg["label_cols"]
print(f"checkpoint: epoch {ck['epoch']} | gold AUC {ck.get('auc_gold', float('nan')):.4f}")
print(f"  {BACKBONE} | {len(PLANES)} planes x {N_SLICES} slices x {IMG}px")

device: cuda Tesla T4
Loading ckpt
checkpoint: epoch 4 | gold AUC 0.8548
  tf_efficientnet_b0.ns_jft_in1k | 3 planes x 16 slices x 224px


## 1 · Decoder check

The compressed transfer syntaxes (JPEG Lossless, JPEG 2000) need a backend beyond
pydicom. This notebook is offline and cannot install one, so find out **now** whether
the image ships it — and if not, which fraction of test series it costs you.

In [2]:
import pydicom, cv2
print("pydicom", pydicom.__version__, "| cv2", cv2.__version__)

backends = {}
for name in ("pylibjpeg", "pylibjpeg_libjpeg", "pylibjpeg_openjpeg", "gdcm", "PIL"):
    try:
        __import__(name); backends[name] = True
    except ImportError:
        backends[name] = False
for k, v in backends.items():
    print(f"  {k:<20} {'yes' if v else 'NO'}")

test_series = pd.read_csv(os.path.join(COMP_DIR, "test_series.csv"))
test_series["series_dir"] = [
    os.path.join(COMP_DIR, "test_series", st, se)
    for st, se in zip(test_series["StudyInstanceUID"], test_series["SeriesInstanceUID"])]

# survey transfer syntaxes and actually try to decode one slice of each
seen = {}
for d in test_series["series_dir"].head(60):
    fs = glob.glob(os.path.join(d, "*.dcm"))
    if not fs:
        continue
    try:
        ds = pydicom.dcmread(fs[0], stop_before_pixels=True)
        ts = str(ds.file_meta.TransferSyntaxUID)
        name = ds.file_meta.TransferSyntaxUID.name
    except Exception as e:
        seen.setdefault("UNREADABLE HEADER", [0, str(e)[:60]])[0] += 1
        continue
    if ts not in seen:
        try:
            pydicom.dcmread(fs[0]).pixel_array
            seen[ts] = [1, f"OK   ({name})"]
        except Exception as e:
            seen[ts] = [1, f"FAIL ({name}): {type(e).__name__}"]
    else:
        seen[ts][0] += 1

print("\ntransfer syntaxes in the example test data:")
for ts, (n, status) in sorted(seen.items(), key=lambda kv: -kv[1][0]):
    print(f"  {n:>4} series  {status}")
if any("FAIL" in s for _, s in seen.values()):
    print("\n!! Some syntaxes cannot be decoded offline. Add pylibjpeg wheels as a")
    print("   dataset and `pip install --no-index --find-links=<dir>` before scoring.")
    print("   Those studies will fall back to the prior, costing real AUC.")

pydicom 3.0.2 | cv2 4.13.0
  pylibjpeg            NO
  pylibjpeg_libjpeg    NO
  pylibjpeg_openjpeg   NO
  gdcm                 NO
  PIL                  yes

transfer syntaxes in the example test data:
    15 series  OK   (Explicit VR Little Endian)


## 2 · Series selection and decoding (identical to step 2)

In [3]:
def _sort_key(ds):
    # Canonicalise direction: the normal's sign depends on how the scanner wrote
    # ImageOrientationPatient, so force its dominant component positive and every
    # stack runs the same anatomical way. Must match step 2 exactly.
    iop = getattr(ds, "ImageOrientationPatient", None)
    ipp = getattr(ds, "ImagePositionPatient", None)
    if iop is not None and ipp is not None:
        try:
            n = np.cross(np.array(iop[:3], float), np.array(iop[3:6], float))
            n = n * np.sign(n[np.argmax(np.abs(n))] or 1.0)
            return float(np.dot(np.array(ipp, float), n))
        except Exception:
            pass
    return float(getattr(ds, "InstanceNumber", 0) or 0)


def load_series(series_dir, n_slices=None, img=None):
    n_slices = n_slices or N_SLICES
    img = img or IMG
    files = glob.glob(os.path.join(series_dir, "*.dcm"))
    if not files:
        raise FileNotFoundError(series_dir)

    keyed = []
    for f in files:
        try:
            keyed.append((_sort_key(pydicom.dcmread(f, stop_before_pixels=True)), f))
        except Exception:
            continue
    if not keyed:
        raise RuntimeError("no readable headers")
    keyed.sort(key=lambda t: t[0])
    ordered = [f for _, f in keyed]

    idx = np.linspace(0, len(ordered) - 1, n_slices).round().astype(int)
    out = np.zeros((n_slices, img, img), dtype=np.float32)
    ok = 0
    for j, i in enumerate(idx):
        try:
            ds = pydicom.dcmread(ordered[i])
            a = ds.pixel_array.astype(np.float32)
            m = float(getattr(ds, "RescaleSlope", 1) or 1)
            b = float(getattr(ds, "RescaleIntercept", 0) or 0)
            if (m, b) != (1.0, 0.0):
                a = a * m + b
            if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
                a = a.max() - a
            if a.ndim == 3:
                a = a.mean(axis=-1)
            out[j] = cv2.resize(a, (img, img), interpolation=cv2.INTER_AREA)
            ok += 1
        except Exception:
            continue
    if ok == 0:
        raise RuntimeError("no decodable slices")

    # whole-volume percentiles: shift/scale invariant, unlike masking to out > 0
    lo, hi = np.percentile(out, [1, 99])
    if hi <= lo:
        lo, hi = float(out.min()), float(out.max())
    if hi <= lo:
        hi = lo + 1.0
    return np.clip((out - lo) / (hi - lo) * 255.0, 0, 255).astype(np.uint8)

In [4]:
test = pd.read_csv(os.path.join(COMP_DIR, "test.csv"))
print(f"test studies: {len(test):,}")

ts = test_series.copy()
ts["Fluid_Sensitive"] = pd.to_numeric(ts["Fluid_Sensitive"], errors="coerce").fillna(0).astype(int)
ts["n_slices"] = [len(glob.glob(os.path.join(d, "*.dcm"))) for d in
                  tqdm(ts["series_dir"], desc="counting slices", unit="series")]

def choose(g):
    picked = {}
    for plane in PLANES:
        cand = g[(g["Anatomical_Plane"] == plane) & (g["n_slices"] >= MIN_SLICES)]
        if not len(cand):
            cand = g[g["Anatomical_Plane"] == plane]
        if len(cand):
            picked[plane] = cand.sort_values(["Fluid_Sensitive", "n_slices"],
                                             ascending=[False, False]).iloc[0]
    return picked

rows = []
for uid, g in ts.groupby("StudyInstanceUID", sort=False):
    picked = choose(g)
    rec = {"StudyInstanceUID": uid}
    for i, plane in enumerate(PLANES):
        s = picked.get(plane)
        rec[f"dir_{i}"] = s["series_dir"] if s is not None else ""
    rows.append(rec)

manifest = test[["StudyInstanceUID"]].merge(pd.DataFrame(rows),
                                            on="StudyInstanceUID", how="left")
for i in range(len(PLANES)):
    manifest[f"dir_{i}"] = manifest[f"dir_{i}"].fillna("")

for i, plane in enumerate(PLANES):
    print(f"  {plane:<9} available for {(manifest[f'dir_{i}'] != '').mean():6.1%} of studies")
print("studies with no series at all:",
      int((manifest[[f"dir_{i}" for i in range(len(PLANES))]] == "").all(axis=1).sum()))

test studies: 3


counting slices:   0%|          | 0/15 [00:00<?, ?series/s]

  Sagittal  available for 100.0% of studies
  Coronal   available for 100.0% of studies
  Axial     available for 100.0% of studies
studies with no series at all: 0


## 3 · Model

In [5]:
import timm

def make_neighbours(x):
    prev = torch.cat([x[:, :, :1], x[:, :, :-1]], dim=2)
    nxt  = torch.cat([x[:, :, 1:], x[:, :, -1:]], dim=2)
    return torch.stack([prev, x, nxt], dim=3)


class AttnPool(nn.Module):
    def __init__(self, dim, hidden=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))
    def forward(self, x, mask=None):
        a = self.score(x).squeeze(-1)
        if mask is not None:
            a = a.masked_fill(~mask, torch.finfo(a.dtype).min)
        a = a.softmax(dim=-1)
        return (x * a.unsqueeze(-1)).sum(1), a


class LabelAttnPool(nn.Module):
    def __init__(self, dim, n_labels):
        super().__init__()
        self.score = nn.Linear(dim, n_labels)
    def forward(self, x, mask):
        a = self.score(x)
        a = a.masked_fill(~mask.unsqueeze(-1), torch.finfo(a.dtype).min)
        a = a.softmax(dim=1)
        return torch.einsum("bsd,bsl->bld", x, a), a


class KneeNet(nn.Module):
    def __init__(self, backbone, n_out, n_planes, pretrained=False, drop=0.3):
        super().__init__()
        self.enc = timm.create_model(backbone, pretrained=pretrained,
                                     num_classes=0, in_chans=3)
        d = self.enc.num_features
        self.slice_pool = AttnPool(d)
        self.plane_emb = nn.Parameter(torch.zeros(n_planes, d))
        self.plane_pool = LabelAttnPool(d, n_out)
        self.drop = nn.Dropout(drop)
        self.head_w = nn.Parameter(torch.zeros(n_out, d))
        self.head_b = nn.Parameter(torch.zeros(n_out))

    def forward(self, x, present):
        with torch.amp.autocast("cuda", enabled=x.is_cuda):
            B, S, T, H, W = x.shape
            x = x.float().div_(255.).sub_(0.449).div_(0.226)
            x3 = make_neighbours(x).reshape(B * S * T, 3, H, W)
            f = self.enc(x3).reshape(B * S, T, -1)
            v, _ = self.slice_pool(f)
            v = v.reshape(B, S, -1) + self.plane_emb.unsqueeze(0)
            m = present.clone()
            m[~m.any(dim=1)] = True
            z, att = self.plane_pool(v, m)
            logits = (self.drop(z) * self.head_w.unsqueeze(0)).sum(-1) + self.head_b
        return logits, att


# pretrained=False: no download, so this works with Internet OFF
model = KneeNet(BACKBONE, len(LABEL_COLS), len(PLANES), pretrained=False).to(DEV)
missing, unexpected = model.load_state_dict(ck["model"], strict=False)
assert not missing and not unexpected, f"state_dict mismatch: {missing} / {unexpected}"
model.eval()
print(f"loaded {sum(p.numel() for p in model.parameters())/1e6:.1f}M params, strict match")

loaded 4.2M params, strict match


## 4 · Inference

In [6]:
class TestDS(Dataset):
    def __init__(self, manifest):
        self.dirs = manifest[[f"dir_{i}" for i in range(len(PLANES))]].to_numpy()

    def __len__(self):
        return len(self.dirs)

    def __getitem__(self, i):
        vol = np.zeros((len(PLANES), N_SLICES, IMG, IMG), dtype=np.uint8)
        present = np.zeros(len(PLANES), dtype=bool)
        for s, d in enumerate(self.dirs[i]):
            if not d:
                continue
            try:
                vol[s] = load_series(d)
                present[s] = True
            except Exception:
                pass                      # leave the slot blank; masked downstream
        return torch.from_numpy(vol), torch.from_numpy(present), i


def collate(b):
    xs, ps, ii = zip(*b)
    return torch.stack(xs), torch.stack(ps), torch.tensor(ii)


dl = DataLoader(TestDS(manifest), batch_size=BATCH, shuffle=False,
                num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)

N = len(manifest)
preds = np.full((N, len(LABEL_COLS)), np.nan, dtype=np.float32)
n_ok = n_blank = 0
t0 = time.time()

with torch.no_grad():
    for x, p, idx in tqdm(dl, desc="inference", unit="batch"):
        if (time.time() - t0) / 3600 > MAX_HOURS:
            print(f"\n!! {MAX_HOURS}h budget hit -- remaining studies get the prior")
            break
        try:
            logit, _ = model(x.to(DEV, non_blocking=True), p.to(DEV))
            preds[idx.numpy()] = torch.sigmoid(logit.float()).cpu().numpy()
            n_ok += int(p.any(dim=1).sum())
            n_blank += int((~p.any(dim=1)).sum())
        except Exception:
            traceback.print_exc()          # keep going: a bad batch must not end the run
            continue

done = ~np.isnan(preds).any(axis=1)
print(f"\npredicted {done.sum():,}/{N:,} studies in {(time.time()-t0)/60:.1f} min")
print(f"  had >=1 decodable series : {n_ok:,}")
print(f"  all planes blank         : {n_blank:,}  (model still ran, on zeros)")

inference:   0%|          | 0/1 [00:00<?, ?batch/s]


predicted 3/3 studies in 0.2 min
  had >=1 decodable series : 3
  all planes blank         : 0  (model still ran, on zeros)


In [7]:
# Fill any gap with the mean of what we did predict -- self-contained, no hardcoded
# prior, and under a ranking metric a constant for a few studies is nearly free.
if done.sum() == 0:
    print("!! nothing predicted; falling back to 0.5 everywhere")
    fill = np.full(len(LABEL_COLS), 0.5, dtype=np.float32)
else:
    fill = preds[done].mean(axis=0)
preds[~done] = fill
print(f"filled {int((~done).sum())} studies with the mean prediction")

print("\npredicted positive rate (mean probability per finding):")
print(pd.Series(preds.mean(axis=0), index=LABEL_COLS).round(3).to_string())

filled 0 studies with the mean prediction

predicted positive rate (mean probability per finding):
ACL                 0.017
MCL                 0.033
Medial Meniscus     0.276
Lateral Meniscus    0.254
Medial OA           0.569
Lateral OA          0.311
PF OA               0.590
Effusion            0.643
Synovitis           0.274
Baker's             0.262
Contusion           0.055
Fracture            0.015


## 5 · Write the submission

In [8]:
sub = pd.DataFrame(preds, columns=LABEL_COLS)
sub.insert(0, "StudyInstanceUID", manifest["StudyInstanceUID"].to_numpy())

# Match sample_submission exactly: same columns, same order, same rows.
sample = pd.read_csv(os.path.join(COMP_DIR, "sample_submission.csv"))
sub = sample[["StudyInstanceUID"]].merge(sub, on="StudyInstanceUID", how="left")
missing = sub[LABEL_COLS].isna().any(axis=1).sum()
if missing:
    print(f"!! {missing} rows missing after the merge -- filling with the prior")
    for k, c in enumerate(LABEL_COLS):
        sub[c] = sub[c].fillna(float(fill[k]))
sub = sub[sample.columns]

assert list(sub.columns) == list(sample.columns), "column mismatch"
assert len(sub) == len(sample), f"row count {len(sub)} != {len(sample)}"
assert sub[LABEL_COLS].isna().sum().sum() == 0, "NaNs left in the submission"
assert sub[LABEL_COLS].to_numpy().min() >= 0 and sub[LABEL_COLS].to_numpy().max() <= 1

sub.to_csv(OUT_CSV, index=False)
print(f"wrote {OUT_CSV}  {sub.shape}")
print(sub.head().round(4).to_string(index=False))

wrote /kaggle/working/submission.csv  (3, 13)
                                                StudyInstanceUID    ACL    MCL  Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA  PF OA  Effusion  Synovitis  Baker's  Contusion  Fracture
1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191 0.0065 0.0465           0.2082            0.0082     0.1976      0.0370 0.2618    0.8365     0.1351   0.1849     0.0831    0.0095
1.2.826.0.1.3680043.8.498.10062861783145312629332250977456991776 0.0443 0.0228           0.2993            0.1882     0.7191      0.4581 0.7035    0.6980     0.5966   0.5726     0.0683    0.0280
1.2.826.0.1.3680043.8.498.10067514707072572280263481548497591402 0.0012 0.0296           0.3207            0.5655     0.7915      0.4377 0.8062    0.3940     0.0897   0.0284     0.0144    0.0066
